In [2]:
# Run this in a new notebook or Python script locally
import pandas as pd
import json

# Check the file structure first
print("Reading first few rows...")
sample = pd.read_csv(
    '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival/data/external/pancancer_expression.xena',
    sep='\t', index_col=0, nrows=5)

print(f"Shape preview: {sample.shape}")
print(f"Index (genes): {list(sample.index[:5])}")
print(f"Columns (patients): {list(sample.columns[:5])}")


Reading first few rows...
Shape preview: (5, 11069)
Index (genes): [100130426, 100133144, 100134869, 10357, 10431]
Columns (patients): ['TCGA-OR-A5J1-01', 'TCGA-OR-A5J2-01', 'TCGA-OR-A5J3-01', 'TCGA-OR-A5J5-01', 'TCGA-OR-A5J6-01']


In [6]:
# Check our gene list format
gene_list = json.load(open(
    '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival/models/before fusion/gene_list.json'))

print(f"Our genes (first 5): {gene_list[:5]}")
print(f"Total genes: {len(gene_list)}")

# Check if pan-cancer index is numeric or string
print(f"\nPan-cancer gene ID type: {type(sample.index[0])}")

Our genes (first 5): ['LINGO2', 'EPGN', 'DKK1', 'CD109', 'LOC441869']
Total genes: 1000

Pan-cancer gene ID type: <class 'numpy.int64'>


In [7]:
# Download gene symbol to Entrez ID mapping using mygene
import subprocess
subprocess.run(['pip', 'install', 'mygene', '-q'])

import mygene
mg = mygene.MyGeneInfo()

print(f"Converting {len(gene_list)} gene symbols to Entrez IDs...")
result = mg.querymany(gene_list, scopes='symbol', fields='entrezgene', species='human')

# Build mapping dictionary
symbol_to_entrez = {}
for r in result:
    if 'entrezgene' in r and 'query' in r:
        symbol_to_entrez[r['query']] = int(r['entrezgene'])

print(f"Successfully mapped: {len(symbol_to_entrez)} / {len(gene_list)} genes")
print(f"Example: EGFR -> {symbol_to_entrez.get('EGFR', 'not found')}")
print(f"Example: TP53 -> {symbol_to_entrez.get('TP53', 'not found')}")

# Save mapping
with open('/Users/parthshringarpure/Desktop/Projects/luad_survival/models/symbol_to_entrez.json', 'w') as f:
    json.dump({k: str(v) for k, v in symbol_to_entrez.items()}, f, indent=2)

print("Saved: models/symbol_to_entrez.json")

Converting 1000 gene symbols to Entrez IDs...


16 input query terms found dup hits:	[('RRN3P1', 2), ('TRPC2', 2), ('NAPSB', 2), ('MST1P2', 2), ('ABCC6P1', 2), ('ABCA17P', 2), ('FAM95B1
169 input query terms found no hit:	['LOC441869', 'BZRAP1', 'GPR172B', 'C20orf197', 'CMAH', 'GYLTL1B', 'C12orf39', 'LOC283070', 'ARNTL2'


Successfully mapped: 831 / 1000 genes
Example: EGFR -> not found
Example: TP53 -> not found
Saved: models/symbol_to_entrez.json


In [8]:
# Better query — request entrezgene explicitly
result = mg.querymany(gene_list, 
                       scopes='symbol', 
                       fields='entrezgene,symbol', 
                       species='human',
                       returnall=True)

symbol_to_entrez = {}
for r in result['out']:
    if 'entrezgene' in r:
        symbol_to_entrez[r['query']] = str(int(float(r['entrezgene'])))

print(f"Successfully mapped: {len(symbol_to_entrez)} / {len(gene_list)} genes")
print(f"EGFR -> {symbol_to_entrez.get('EGFR', 'not found')}")
print(f"TP53 -> {symbol_to_entrez.get('TP53', 'not found')}")
print(f"LINGO2 -> {symbol_to_entrez.get('LINGO2', 'not found')}")

# Check how many of our mapped entrez IDs exist in pan-cancer data
print(f"\nChecking overlap with pan-cancer dataset...")
pancancer_genes = set(str(g) for g in sample.index)
our_entrez = set(symbol_to_entrez.values())
overlap = our_entrez.intersection(pancancer_genes)
print(f"Our mapped genes in pan-cancer: {len(overlap)} / {len(symbol_to_entrez)}")

16 input query terms found dup hits:	[('RRN3P1', 2), ('TRPC2', 2), ('NAPSB', 2), ('MST1P2', 2), ('ABCC6P1', 2), ('ABCA17P', 2), ('FAM95B1
169 input query terms found no hit:	['LOC441869', 'BZRAP1', 'GPR172B', 'C20orf197', 'CMAH', 'GYLTL1B', 'C12orf39', 'LOC283070', 'ARNTL2'


Successfully mapped: 831 / 1000 genes
EGFR -> not found
TP53 -> not found
LINGO2 -> 158038

Checking overlap with pan-cancer dataset...
Our mapped genes in pan-cancer: 0 / 831


In [12]:
# Read just the index column to check gene IDs
pancancer_genes_check = pd.read_csv(
    '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival/data/external/pancancer_expression.xena',
    sep='\t', index_col=0, nrows=20)

print(f"First 20 gene IDs: {list(pancancer_genes_check.index)}")
print(f"ID type: {type(pancancer_genes_check.index[0])}")

# Check if common gene entrez IDs are present
# EGFR=1956, TP53=7157, KRAS=3845
for name, eid in [('EGFR', 1956), ('TP53', 7157), ('KRAS', 3845)]:
    print(f"  {name} (entrez {eid}) in index? {eid in pancancer_genes_check.index}")

First 20 gene IDs: [100130426, 100133144, 100134869, 10357, 10431, 136542, 155060, 26823, 280660, 317712, 340602, 388795, 390284, 391343, 391714, 404770, 441362, 442388, 553137, 57714]
ID type: <class 'numpy.int64'>
  EGFR (entrez 1956) in index? False
  TP53 (entrez 7157) in index? False
  KRAS (entrez 3845) in index? False
